# Análise de Dados — Viagens a Serviço do Governo Federal

## 1. Objetivo

Este notebook concentra a apresentação da Fase 3 do projeto, reutilizando a lógica analítica já validada para:

- validar a camada Gold;
- responder às quatro perguntas de negócio;
- exibir consultas SQL;
- apresentar resultados em tabelas;
- gerar gráficos;
- registrar análises escritas.

A implementação reaproveita a lógica existente em `sql/3_criar_gold.sql` e `src/3_analisar.py`, preservando os resultados já validados.

In [1]:
import sys
from pathlib import Path
import importlib.util

import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)


def localizar_raiz_projeto() -> Path:
    candidatos = [Path.cwd(), Path.cwd().parent]

    for candidato in candidatos:
        if (
            (candidato / "src" / "3_analisar.py").exists()
            and (candidato / "sql" / "3_criar_gold.sql").exists()
        ):
            return candidato

    raise RuntimeError("Não foi possível localizar a raiz do projeto.")


ROOT = localizar_raiz_projeto()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.banco import conectar


def carregar_modulo(caminho: Path, nome: str):
    spec = importlib.util.spec_from_file_location(nome, caminho)
    modulo = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(modulo)
    return modulo


analise = carregar_modulo(ROOT / "src" / "3_analisar.py", "analise_fase3")
gold_sql = (ROOT / "sql" / "3_criar_gold.sql").read_text(encoding="utf-8")


def consultar_dataframe(sql_texto: str) -> pd.DataFrame:
    conexao = conectar()
    try:
        with conexao.cursor() as cursor:
            cursor.execute(sql_texto)
            registros = cursor.fetchall()
            colunas = [descricao[0] for descricao in cursor.description]
        return pd.DataFrame(registros, columns=colunas)
    finally:
        conexao.close()


def verificar_objetos_gold() -> None:
    conexao = conectar()
    try:
        with conexao.cursor() as cursor:
            cursor.execute(
                """
                SELECT EXISTS (
                    SELECT 1
                    FROM information_schema.tables
                    WHERE table_schema = current_schema()
                      AND table_name = 'gold_resumo_pagamentos_mensais'
                ) AS tabela_gold_existe;
                """
            )
            tabela_gold_existe = bool(cursor.fetchone()[0])

            cursor.execute(
                """
                SELECT EXISTS (
                    SELECT 1
                    FROM information_schema.views
                    WHERE table_schema = current_schema()
                      AND table_name = 'vw_resumo_pagamentos_mensais'
                ) AS view_gold_existe;
                """
            )
            view_gold_existe = bool(cursor.fetchone()[0])
    finally:
        conexao.close()

    if tabela_gold_existe and view_gold_existe:
        print("Camada Gold encontrada e pronta para an?lise.")
        return

    raise RuntimeError(
        "A camada Gold n?o est? dispon?vel. Execute primeiro "
        "sql/3_criar_gold.sql no PostgreSQL e depois execute novamente o notebook."
    )


def mostrar_sql(sql_texto: str) -> None:
    display(Markdown("```sql\n" + sql_texto + "\n```"))


def formatar_inteiro_brasileiro(valor) -> str:
    if valor is None or pd.isna(valor):
        return ""
    return f"{int(valor):,}".replace(",", ".")


def formatar_decimal_brasileiro(valor) -> str:
    if valor is None or pd.isna(valor):
        return ""
    return f"{float(valor):.2f}".replace(".", ",")


def formatar_percentual_brasileiro(valor) -> str:
    if valor is None or pd.isna(valor):
        return ""
    return f"{float(valor):.2f}%".replace(".", ",")


def preparar_exibicao_pergunta_1(df: pd.DataFrame) -> pd.DataFrame:
    df_exibicao = df.copy()
    df_exibicao["qtd_viagens"] = df_exibicao["qtd_viagens"].map(formatar_inteiro_brasileiro)
    df_exibicao["duracao_media_dias"] = df_exibicao["duracao_media_dias"].map(formatar_decimal_brasileiro)
    df_exibicao["valor_total_medio"] = df_exibicao["valor_total_medio"].map(analise.formatar_reais)
    df_exibicao["custo_medio_diario"] = df_exibicao["custo_medio_diario"].map(analise.formatar_reais)
    return df_exibicao


def preparar_exibicao_pergunta_2(df: pd.DataFrame) -> pd.DataFrame:
    df_exibicao = df.copy()
    df_exibicao["qtd_viagens"] = df_exibicao["qtd_viagens"].map(formatar_inteiro_brasileiro)
    df_exibicao["custo_medio_diario"] = df_exibicao["custo_medio_diario"].map(analise.formatar_reais)
    df_exibicao["valor_total_medio"] = df_exibicao["valor_total_medio"].map(analise.formatar_reais)
    return df_exibicao


def preparar_exibicao_pergunta_3(df: pd.DataFrame) -> pd.DataFrame:
    df_exibicao = df.copy()
    df_exibicao["ano_referencia"] = df_exibicao["ano_referencia"].map(
        lambda valor: str(int(valor)) if pd.notna(valor) else ""
    )
    df_exibicao["mes_referencia"] = df_exibicao["mes_referencia"].map(
        lambda valor: f"{int(valor):02d}" if pd.notna(valor) else ""
    )
    df_exibicao["valor_total_pago"] = df_exibicao["valor_total_pago"].map(analise.formatar_reais)
    df_exibicao["participacao_percentual"] = df_exibicao["participacao_percentual"].map(formatar_percentual_brasileiro)
    return df_exibicao


def preparar_exibicao_pergunta_4(df: pd.DataFrame) -> pd.DataFrame:
    df_exibicao = df.copy()
    df_exibicao["valor_total_pago"] = df_exibicao["valor_total_pago"].map(analise.formatar_reais)
    df_exibicao["qtd_pagamentos"] = df_exibicao["qtd_pagamentos"].map(formatar_inteiro_brasileiro)
    df_exibicao["valor_medio_pagamento"] = df_exibicao["valor_medio_pagamento"].map(analise.formatar_reais)
    df_exibicao["qtd_viagens_atendidas"] = df_exibicao["qtd_viagens_atendidas"].map(formatar_inteiro_brasileiro)
    return df_exibicao


print("Recursos carregados com sucesso.")

Recursos carregados com sucesso.


## 2. Conexão com o PostgreSQL

A conexão reutiliza a configuração existente do projeto, carregada a partir do arquivo `.env` por meio de `src/config.py` e `src/banco.py`. Nenhuma credencial é registrada neste notebook.

In [2]:
conexao = conectar()

try:
    with conexao.cursor() as cursor:
        cursor.execute("SELECT current_database() AS banco;")
        dados_conexao = cursor.fetchall()
        colunas = [descricao[0] for descricao in cursor.description]
finally:
    conexao.close()

df_conexao = pd.DataFrame(dados_conexao, columns=colunas)
display(df_conexao)

banco
etl_viagens_governo


## 3. Camada Gold

### 3.1 Criação da tabela Gold

### 3.2 Criação da View Gold

A criação e a materialização da camada Gold são realizadas pelo arquivo `sql/3_criar_gold.sql`. Neste notebook, a estrutura já criada é consultada e validada antes das análises.

O SQL é exibido abaixo apenas como referência da etapa analítica. Ele não é executado automaticamente por este notebook.

In [3]:
mostrar_sql(gold_sql)

```sql
-- CRIACAO DA CAMADA GOLD

-- REMOCAO DE OBJETOS ANTERIORES
BEGIN;

DROP VIEW IF EXISTS vw_resumo_pagamentos_mensais;
DROP TABLE IF EXISTS gold_resumo_pagamentos_mensais;

-- CRIACAO DA TABELA GOLD
CREATE TABLE gold_resumo_pagamentos_mensais (
    ano_referencia INTEGER,
    mes_referencia INTEGER,
    nome_orgao_pagador VARCHAR(255),
    tipo_pagamento VARCHAR(50),
    qtd_viagens INTEGER,
    qtd_pagamentos INTEGER,
    valor_total_pago DECIMAL(14,2),
    valor_medio_pagamento DECIMAL(12,2)
);

-- CARGA DA GOLD
INSERT INTO gold_resumo_pagamentos_mensais (
    ano_referencia,
    mes_referencia,
    nome_orgao_pagador,
    tipo_pagamento,
    qtd_viagens,
    qtd_pagamentos,
    valor_total_pago,
    valor_medio_pagamento
)
SELECT
    EXTRACT(YEAR FROM v.data_inicio)::INTEGER AS ano_referencia,
    EXTRACT(MONTH FROM v.data_inicio)::INTEGER AS mes_referencia,
    p.nome_orgao_pagador,
    p.tipo_pagamento,
    COUNT(DISTINCT v.id_viagem)::INTEGER AS qtd_viagens,
    COUNT(p.id_

### 3.3 Validação da Gold

Antes das análises, o notebook verifica se a tabela `gold_resumo_pagamentos_mensais` e a view `vw_resumo_pagamentos_mensais` já existem no PostgreSQL. Caso não existam, a execução deve ser interrompida e a camada Gold deve ser criada previamente com `sql/3_criar_gold.sql`.

Depois da verificação de existência, são preservadas as validações já aprovadas para:

- quantidade de registros da tabela Gold;
- quantidade de registros da View;
- valor total da Gold;
- valor correspondente na Silver;
- diferenças entre tabela e View usando `EXCEPT` nos dois sentidos.

In [4]:
verificar_objetos_gold()

Camada Gold encontrada e pronta para an?lise.


In [5]:
SQL_VALIDACAO_GOLD = {
    "Quantidade de registros da tabela Gold": (
        "SELECT COUNT(*) AS qtd_registros_tabela_gold "
        "FROM gold_resumo_pagamentos_mensais;"
    ),
    "Quantidade de registros da View Gold": (
        "SELECT COUNT(*) AS qtd_registros_view_gold "
        "FROM vw_resumo_pagamentos_mensais;"
    ),
    "Valor total da Gold": (
        "SELECT SUM(valor_total_pago) AS total_pago_gold "
        "FROM gold_resumo_pagamentos_mensais;"
    ),
    "Valor correspondente na Silver": """
        SELECT SUM(p.valor) AS total_pago_silver
        FROM silver_pagamento p
        INNER JOIN silver_viagem v
            ON v.id_viagem = p.id_viagem
        WHERE v.data_inicio IS NOT NULL
          AND p.nome_orgao_pagador IS NOT NULL
          AND p.tipo_pagamento IS NOT NULL;
    """.strip(),
    "Diferen?as entre tabela e View": """
        SELECT COUNT(*) AS qtd_diferencas
        FROM (
            (
                SELECT * FROM gold_resumo_pagamentos_mensais
                EXCEPT
                SELECT * FROM vw_resumo_pagamentos_mensais
            )
            UNION ALL
            (
                SELECT * FROM vw_resumo_pagamentos_mensais
                EXCEPT
                SELECT * FROM gold_resumo_pagamentos_mensais
            )
        ) divergencias;
    """.strip(),
}

for titulo, consulta in SQL_VALIDACAO_GOLD.items():
    display(Markdown(f"**{titulo}**"))
    mostrar_sql(consulta)
    df_validacao = consultar_dataframe(consulta)
    display(df_validacao)

**Quantidade de registros da tabela Gold**

```sql
SELECT COUNT(*) AS qtd_registros_tabela_gold FROM gold_resumo_pagamentos_mensais;
```

qtd_registros_tabela_gold
3348


**Quantidade de registros da View Gold**

```sql
SELECT COUNT(*) AS qtd_registros_view_gold FROM vw_resumo_pagamentos_mensais;
```

...

```sql
SELECT SUM(p.valor) AS total_pago_silver
        FROM silver_pagamento p
        INNER JOIN silver_viagem v
            ON v.id_viagem = p.id_viagem
        WHERE v.data_inicio IS NOT NULL
          AND p.nome_orgao_pagador IS NOT NULL
          AND p.tipo_pagamento IS NOT NULL;
```

total_pago_silver
1194365457.37


**Diferen?as entre tabela e View**

```sql
SELECT COUNT(*) AS qtd_diferencas
        FROM (
            (
                SELECT * FROM gold_resumo_pagamentos_mensais
                EXCEPT
                SELECT * FROM vw_resumo_pagamentos_mensais
            )
            UNION ALL
            (
                SELECT * FROM vw_resumo_pagamentos_mensais
                EXCEPT
                SELECT * FROM gold_resumo_pagamentos_mensais
            )
        ) divergencias;
```

qtd_diferencas
0


## 4. Pergunta 1

**Pergunta:** Quais viagens urgentes custam mais por dia do que as não urgentes?

**Fonte dos dados:** `silver_viagem`

**Métricas:**

- quantidade de viagens;
- duração média em dias;
- valor total médio;
- custo médio diário.

A implementação abaixo reutiliza integralmente a lógica validada em `src/3_analisar.py`.

### Consulta SQL

In [6]:
mostrar_sql(analise.SQL_PERGUNTA_1)

```sql
SELECT
    viagem_urgente,
    COUNT(*)::INTEGER AS qtd_viagens,
    AVG(duracao_dias)::DECIMAL(12,2) AS duracao_media_dias,
    AVG(valor_total)::DECIMAL(14,2) AS valor_total_medio,
    AVG(valor_total / NULLIF(duracao_dias, 0))::DECIMAL(14,2) AS custo_medio_diario
FROM silver_viagem
WHERE duracao_dias IS NOT NULL
  AND duracao_dias > 0
  AND valor_total IS NOT NULL
  AND viagem_urgente IS NOT NULL
GROUP BY viagem_urgente
ORDER BY viagem_urgente;
```

### Resultado

In [7]:
df_pergunta_1 = analise.consultar_pergunta_1()
analise.validar_resultado_pergunta_1(df_pergunta_1)
df_exibicao_pergunta_1 = preparar_exibicao_pergunta_1(df_pergunta_1)
display(df_exibicao_pergunta_1)

viagem_urgente,qtd_viagens,duracao_media_dias,valor_total_medio,custo_medio_diario
NÃO,137.911,"7,31","R$ 2.954,75","R$ 441,94"
SIM,203.949,"8,57","R$ 3.827,69","R$ 597,08"


### Gráfico

In [8]:
analise.gerar_grafico_pergunta_1(df_pergunta_1)
print("Gr?fico salvo em: ../output/pergunta_1_viagens_urgentes.png")

Gr?fico salvo em: ../output/pergunta_1_viagens_urgentes.png


![Custo médio diário por urgência](../output/pergunta_1_viagens_urgentes.png)

### Análise

A análise abaixo é reutilizada diretamente de `src/3_analisar.py`, preservando a lógica analítica validada do projeto.

In [9]:
display(Markdown(analise.analisar_pergunta_1(df_pergunta_1)))

O grupo com maior custo medio diario foi o urgente (SIM). As viagens urgentes registraram custo medio diario de R$ 597,08, contra R$ 441,94 nas nao urgentes, uma diferenca absoluta de R$ 155,14 e diferenca percentual de 35.10%. Em volume, o grupo urgente somou 203949 viagens e o nao urgente 137911, com participacao de 59.66% das urgentes no total analisado. A duracao media foi de 8.57 dias nas urgentes e 7.31 dias nas nao urgentes, enquanto o valor total medio foi de R$ 3.827,69 e R$ 2.954,75, respectivamente. Sim. A urgencia encarece o dia da viagem. As viagens urgentes representam 59.66% das viagens analisadas e, portanto, constituem a maioria do conjunto, com volume relevante.

## 5. Pergunta 2

**Pergunta:** Como o custo médio diário varia conforme a duração da viagem?

**Fonte dos dados:** `silver_viagem`

**Métricas e faixas:**

- `1 dia`;
- `2 a 3 dias`;
- `4 a 7 dias`;
- `8 a 15 dias`;
- `Acima de 15 dias`.

A definição de `duracao_dias` permanece a da camada Silver: `(data_fim - data_inicio) + 1`.

### Consulta SQL

In [10]:
mostrar_sql(analise.SQL_PERGUNTA_2)

```sql
SELECT
    faixa_duracao,
    COUNT(*)::INTEGER AS qtd_viagens,
    AVG(valor_total / NULLIF(duracao_dias, 0))::DECIMAL(14,2) AS custo_medio_diario,
    AVG(valor_total)::DECIMAL(14,2) AS valor_total_medio
FROM (
    SELECT
        CASE
            WHEN duracao_dias = 1 THEN '1 dia'
            WHEN duracao_dias BETWEEN 2 AND 3 THEN '2 a 3 dias'
            WHEN duracao_dias BETWEEN 4 AND 7 THEN '4 a 7 dias'
            WHEN duracao_dias BETWEEN 8 AND 15 THEN '8 a 15 dias'
            WHEN duracao_dias > 15 THEN 'Acima de 15 dias'
        END AS faixa_duracao,
        duracao_dias,
        valor_total
    FROM silver_viagem
    WHERE duracao_dias IS NOT NULL
      AND duracao_dias > 0
      AND valor_total IS NOT NULL
) AS viagens_classificadas
WHERE faixa_duracao IS NOT NULL
GROUP BY faixa_duracao
ORDER BY
    CASE faixa_duracao
        WHEN '1 dia' THEN 1
        WHEN '2 a 3 dias' THEN 2
        WHEN '4 a 7 dias' THEN 3
        WHEN '8 a 15 dias' THEN 4
        WHEN 'Acima de 

### Resultado

In [11]:
df_pergunta_2 = analise.consultar_pergunta_2()
analise.validar_resultado_pergunta_2(df_pergunta_2)
df_exibicao_pergunta_2 = preparar_exibicao_pergunta_2(df_pergunta_2)
display(df_exibicao_pergunta_2)

faixa_duracao,qtd_viagens,custo_medio_diario,valor_total_medio
1 dia,53.979,"R$ 502,04","R$ 502,04"
2 a 3 dias,108.236,"R$ 577,66","R$ 1.440,86"
4 a 7 dias,121.542,"R$ 546,21","R$ 2.793,29"
8 a 15 dias,33.874,"R$ 544,34","R$ 5.704,45"
Acima de 15 dias,24.229,"R$ 341,50","R$ 19.495,56"


### Gráfico

In [12]:
analise.gerar_grafico_pergunta_2(df_pergunta_2)
print("Gr?fico salvo em: ../output/pergunta_2_duracao_viagem.png")

Gr?fico salvo em: ../output/pergunta_2_duracao_viagem.png


![Custo médio diário por duração](../output/pergunta_2_duracao_viagem.png)

### Análise

A análise abaixo é reutilizada diretamente de `src/3_analisar.py`, preservando a lógica analítica validada do projeto.

In [13]:
display(Markdown(analise.analisar_pergunta_2(df_pergunta_2)))

A faixa com maior custo medio diario foi '2 a 3 dias', com R$ 577,66 e 108236 viagens, enquanto a menor foi 'Acima de 15 dias', com R$ 341,50 e 24229 viagens. A diferenca absoluta entre essas extremidades foi de R$ 236,16 e a diferenca percentual foi de 69.15%. A faixa com maior volume de viagens foi '4 a 7 dias', com 121542 registros. Em valor total medio, a faixa mais alta foi 'Acima de 15 dias' com R$ 19.495,56, enquanto a menor foi '1 dia' com R$ 502,04. Assim, o custo medio diario varia por faixa de duracao de modo que o custo medio diario termina menor nas faixas longas, mas sem trajetoria monotonicamente decrescente.

## 6. Pergunta 3

**Pergunta:** Como o valor pago evoluiu mês a mês e qual tipo de pagamento sustenta essa evolução?

**Fonte dos dados:** `gold_resumo_pagamentos_mensais`

**Métricas:**

- valor total pago por mês e tipo de pagamento;
- participação percentual de cada tipo dentro do mês.

O gráfico permanece em linhas, com meses no eixo X e valores monetários legíveis no eixo Y.

### Consulta SQL

In [14]:
mostrar_sql(analise.SQL_PERGUNTA_3)

```sql
WITH pagamentos_mensais AS (
    SELECT
        ano_referencia,
        mes_referencia,
        tipo_pagamento,
        SUM(valor_total_pago) AS valor_total_pago
    FROM gold_resumo_pagamentos_mensais
    WHERE ano_referencia IS NOT NULL
      AND mes_referencia IS NOT NULL
      AND tipo_pagamento IS NOT NULL
    GROUP BY
        ano_referencia,
        mes_referencia,
        tipo_pagamento
)
SELECT
    ano_referencia,
    mes_referencia,
    tipo_pagamento,
    valor_total_pago::DECIMAL(14,2) AS valor_total_pago,
    (
        (
            valor_total_pago
            / NULLIF(
                SUM(valor_total_pago) OVER (
                    PARTITION BY ano_referencia, mes_referencia
                ),
                0
            )
        ) * 100
    )::DECIMAL(7,2) AS participacao_percentual
FROM pagamentos_mensais
ORDER BY
    ano_referencia,
    mes_referencia,
    tipo_pagamento;
```

### Resultado

In [15]:
df_pergunta_3 = analise.consultar_pergunta_3()
analise.validar_resultado_pergunta_3(df_pergunta_3)
df_exibicao_pergunta_3 = preparar_exibicao_pergunta_3(df_pergunta_3)
display(df_exibicao_pergunta_3)

ano_referencia,mes_referencia,tipo_pagamento,valor_total_pago,participacao_percentual
2025,01,DIÁRIAS,"R$ 256.453.682,29","88,81%"
2025,01,PASSAGEM,"R$ 31.847.615,00","11,03%"
2025,01,RESTITUIÇÃO,"R$ 275.735,47","0,10%"
2025,01,Serviço correlato: seguro,"R$ 174.411,14","0,06%"
2025,02,DIÁRIAS,"R$ 74.326.701,66","62,35%"
2025,02,PASSAGEM,"R$ 44.310.096,00","37,17%"
2025,02,RESTITUIÇÃO,"R$ 313.570,21","0,26%"
2025,02,Serviço correlato: seguro,"R$ 259.941,87","0,22%"
2025,03,DIÁRIAS,"R$ 136.196.550,33","66,59%"
2025,03,PASSAGEM,"R$ 67.379.292,96","32,95%"


### Gráfico

In [16]:
analise.gerar_grafico_pergunta_3(df_pergunta_3)
print("Gr?fico salvo em: ../output/pergunta_3_evolucao_pagamentos.png")

Gr?fico salvo em: ../output/pergunta_3_evolucao_pagamentos.png


![Evolução mensal dos pagamentos](../output/pergunta_3_evolucao_pagamentos.png)

### Análise

A análise abaixo é reutilizada diretamente de `src/3_analisar.py`, preservando a lógica analítica validada do projeto.

In [17]:
display(Markdown(analise.analisar_pergunta_3(df_pergunta_3)))

O valor total pago evoluiu ao longo de 6 meses (01/2025, 02/2025, 03/2025, 04/2025, 05/2025, 06/2025), saindo de R$ 288.751.443,90 em 01/2025 para R$ 203.937.334,69 em 06/2025, com variacao de -29.37% entre o primeiro e o ultimo mes. O maior valor mensal ocorreu em 01/2025 (R$ 288.751.443,90) e o menor em 02/2025 (R$ 119.210.309,74). Quanto a sazonalidade, observa-se variacao mensal no periodo, com picos e reducoes descritivas entre meses, mas sem base suficiente para afirmar sazonalidade recorrente; Os dados permitem identificar variacoes mensais, mas o periodo analisado e insuficiente para confirmar um padrao sazonal recorrente. No inicio do periodo, o tipo com maior participacao foi 'DIÁRIAS', com 88.81%. No final do periodo, o maior foi 'DIÁRIAS', com 60.52%. O tipo que mais ganhou participacao entre 01/2025 e 06/2025 foi 'PASSAGEM', com +27.90 pontos percentuais, enquanto o que mais perdeu foi 'DIÁRIAS', com -28.29 pontos percentuais. No periodo analisado, o tipo 'DIÁRIAS' concent

## 7. Pergunta 4

**Pergunta:** Qual é o perfil de gasto dos órgãos pagadores?

**Fonte dos dados:** `gold_resumo_pagamentos_mensais`

**Métricas:**

- valor total pago;
- quantidade de pagamentos;
- valor médio por pagamento;
- quantidade de viagens atendidas.

O gráfico de bolhas preserva:

- eixo X = quantidade de pagamentos;
- eixo Y = valor médio por pagamento;
- tamanho da bolha = valor total pago.

A classificação relativa continua baseada nas medianas do Top 10 por valor total pago.

### Consulta SQL

In [18]:
mostrar_sql(analise.SQL_PERGUNTA_4)

```sql
SELECT
    nome_orgao_pagador,
    SUM(valor_total_pago)::DECIMAL(16,2) AS valor_total_pago,
    SUM(qtd_pagamentos)::INTEGER AS qtd_pagamentos,
    (
        SUM(valor_total_pago)
        / NULLIF(SUM(qtd_pagamentos), 0)
    )::DECIMAL(14,2) AS valor_medio_pagamento,
    SUM(qtd_viagens)::INTEGER AS qtd_viagens_atendidas
FROM gold_resumo_pagamentos_mensais
WHERE nome_orgao_pagador IS NOT NULL
GROUP BY nome_orgao_pagador
ORDER BY valor_total_pago DESC
LIMIT 10;
```

### Resultado

In [19]:
df_pergunta_4 = analise.consultar_pergunta_4()
analise.validar_resultado_pergunta_4(df_pergunta_4)
df_exibicao_pergunta_4 = preparar_exibicao_pergunta_4(df_pergunta_4)
display(df_exibicao_pergunta_4)

nome_orgao_pagador,valor_total_pago,qtd_pagamentos,valor_medio_pagamento,qtd_viagens_atendidas
Fundo Nacional de Segurança Pública,"R$ 278.481.047,89",79.816,"R$ 3.489,04",27.748
Sigiloso,"R$ 200.484.801,68",93.141,"R$ 2.152,49",62.400
Comando da Aeronáutica,"R$ 81.769.144,77",46.193,"R$ 1.770,16",33.692
Instituto Nacional do Seguro Social,"R$ 37.427.601,45",18.324,"R$ 2.042,55",11.742
Comando do Exército,"R$ 36.872.643,95",22.837,"R$ 1.614,60",17.337
Ministério da Gestão e da Inovação em Serviços Públicos - Unidades com vínculo direto,"R$ 35.541.760,71",20.291,"R$ 1.751,60",12.163
Instituto Brasileiro do Meio Ambiente e dos Recursos Naturais Renováveis,"R$ 31.589.853,15",16.758,"R$ 1.885,06",11.194
Ministério das Relações Exteriores - Unidades com vínculo direto,"R$ 25.605.376,38",3.705,"R$ 6.911,03",2.782
Receita Federal do Brasil,"R$ 23.811.027,00",18.917,"R$ 1.258,71",14.601
Ministério da Agricultura e Pecuária - Unidades com vínculo direto,"R$ 22.899.880,25",15.864,"R$ 1.443,51",13.406


### Gráfico

In [20]:
analise.gerar_grafico_pergunta_4(df_pergunta_4)
print("Gr?fico salvo em: ../output/pergunta_4_perfil_orgaos_pagadores.png")

Gr?fico salvo em: ../output/pergunta_4_perfil_orgaos_pagadores.png


![Perfil de gasto dos órgãos pagadores](../output/pergunta_4_perfil_orgaos_pagadores.png)

### Análise

A análise abaixo é reutilizada diretamente de `src/3_analisar.py`, preservando a lógica analítica validada do projeto.

In [21]:
display(Markdown(analise.analisar_pergunta_4(df_pergunta_4)))

No Top 10 por valor total pago, o orgao com maior desembolso foi 'Fundo Nacional de Segurança Pública', com R$ 278.481.047,89, 79816 pagamentos, ticket medio de R$ 3.489,04, 27748 viagens atendidas e perfil relativo 'alto volume / ticket alto'. O maior volume de pagamentos foi de 'Sigiloso', com 93141 pagamentos, enquanto o maior ticket medio apareceu em 'Ministério das Relações Exteriores - Unidades com vínculo direto', com R$ 6.911,03. O maior volume de viagens atendidas ficou com 'Sigiloso', com 62400 viagens. Dentro do Top 10, o menor ticket medio foi o de 'Receita Federal do Brasil', com R$ 1.258,71. A diferenca entre o maior e o menor ticket medio foi de R$ 5.652,32 e 449.06%. As medianas usadas para classificar os perfis relativos dos 10 maiores orgaos foram 19604.0 pagamentos e R$ 1.827,61 de ticket medio. No Top 10, a classificacao relativa pelas medianas indica 2 orgaos em alto volume / ticket alto, 3 em alto volume / ticket baixo, 3 em baixo volume / ticket alto e 2 em baixo

## 8. Conclusões

A Fase 3 demonstra a consolidação da camada Gold e sua aplicação em análises de negócio sobre gastos com viagens a serviço. As quatro perguntas mantêm os resultados já validados e permitem observar, de forma integrada:

- o maior custo médio diário das viagens urgentes;
- a variação do custo médio diário conforme a duração da viagem;
- a evolução mensal dos pagamentos e o papel predominante de `DIÁRIAS` no período analisado;
- os diferentes perfis de gasto entre os principais órgãos pagadores.

Com isso, o notebook organiza a etapa analítica no formato exigido pelo edital, sem alterar a lógica previamente validada no projeto.